# 💳 Notebook 04 — Credit Analysis
**BDM Capstone Project — IIT Madras**

| | |
|---|---|
| **Student** | Namra Sania |
| **Roll No** | 23f2000313 |
| **Business** | Raj Kishore Gupta and Sons, Mubarakpur, Azamgarh, Uttar Pradesh |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

credit_df = pd.read_csv('../data/credit_data.csv')
credit_df['Transaction_Date'] = pd.to_datetime(credit_df['Transaction_Date'], dayfirst=True)
print('Credit data loaded:', credit_df.shape)
print('Customers:', credit_df['Customer_Name'].unique())

## 1. Credit Formula

```
Interest Cost    = (4/100) × Total Credit        [4% per month opportunity cost]
Profit on Credit = (35/100) × (60/100) × Total Credit
Earnings         = Total Repayment + Profit − Total Credit − Interest Cost
```

## 2. Customer-wise Credit Summary

In [ ]:
# Exclude opening balance row (Jan 15)
txn = credit_df[credit_df['Transaction_Date'] > '2024-01-15'].copy()

summary = txn.groupby('Customer_Name').agg(
    Total_Credit    = ('Credit_Amount', 'sum'),
    Total_Repayment = ('Repayment', 'sum')
).reset_index()

INTEREST_RATE = 0.04
MARGIN        = 0.35

summary['Interest_Cost']     = (INTEREST_RATE * summary['Total_Credit']).round(1)
summary['Profit_on_Credit']  = (MARGIN * 0.60 * summary['Total_Credit']).round(1)
summary['Earnings']          = (
    summary['Total_Repayment'] +
    summary['Profit_on_Credit'] -
    summary['Total_Credit'] -
    summary['Interest_Cost']
).round(1)
summary['Status'] = summary['Earnings'].apply(lambda x: '✅ Profit' if x >= 0 else '❌ LOSS')

print('=== CREDIT SUMMARY (35% Margin | 4% Interest Rate) ===')
print(summary.to_string(index=False))
print(f'\nTotal Earnings from all customers: ₹{summary["Earnings"].sum():,.1f}')

## 3. Credit Distribution — Circular Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#1d4ed8','#2563eb','#3b82f6','#60a5fa','#93c5fd','#bfdbfe']
bars = ax.bar(summary['Customer_Name'], summary['Total_Credit'],
              color=colors, edgecolor='white', width=0.5)
ax.set_title('Total Credit Given per Customer\nRaj Kishore Gupta & Sons, Mubarakpur Azamgarh',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Total Credit (₹)')
ax.set_xlabel('Customer')
for bar, val in zip(bars, summary['Total_Credit']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'₹{val:,}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('../dashboards/credit_distribution.png', dpi=150)
plt.show()

## 4. Earnings per Customer

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#16a34a' if e >= 0 else '#dc2626' for e in summary['Earnings']]
bars = ax.bar(summary['Customer_Name'], summary['Earnings'], color=colors, edgecolor='white', width=0.5)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_title('Earnings per Credit Customer (35% Margin, 4% Interest)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Earnings (₹)')
for bar, val in zip(bars, summary['Earnings']):
    ypos = val + 80 if val >= 0 else val - 200
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'₹{val:,.0f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('../dashboards/credit_earnings.png', dpi=150)
plt.show()
print('⚠️  Kishore shows NEGATIVE earnings — problematic customer!')

## 5. Kishore's Credit Activity — Detailed Analysis

In [ ]:
kishore = credit_df[
    (credit_df['Customer_Name'] == 'Kishore') &
    (credit_df['Transaction_Date'] > '2024-01-15')
].sort_values('Transaction_Date').copy()

print('Kishore Full Transaction History:')
print(kishore[['Transaction_Date','Credit_Amount','Repayment','Outstanding_Balance']].to_string(index=False))

# Highlight the problematic Feb 6-14 period
problem = kishore[
    (kishore['Transaction_Date'] >= '2024-02-06') &
    (kishore['Transaction_Date'] <= '2024-02-14')
]
print(f'\n⚠️  Feb 6–14 Repayments (should be ~₹1000+/day):')
print(problem[['Transaction_Date','Credit_Amount','Repayment','Outstanding_Balance']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(kishore['Transaction_Date'], kishore['Credit_Amount'],
       label='Credit Given', color='#ef4444', alpha=0.7, width=0.6)
ax.bar(kishore['Transaction_Date'], kishore['Repayment'],
       label='Repayment', color='#16a34a', alpha=0.7, width=0.6)
ax2 = ax.twinx()
ax2.plot(kishore['Transaction_Date'], kishore['Outstanding_Balance'],
         color='#1d4ed8', linewidth=2, marker='o', markersize=4, label='Outstanding Balance')

# Shade problem period
ax.axvspan(pd.Timestamp('2024-02-06'), pd.Timestamp('2024-02-14'),
           alpha=0.15, color='red', label='Problem Period (Feb 6-14)')

ax.set_title("Kishore's Credit Activity — Skipped Repayments (Feb 6–14)",
             fontsize=12, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Amount (₹)')
ax2.set_ylabel('Outstanding Balance (₹)', color='#1d4ed8')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('../dashboards/kishore_activity.png', dpi=150)
plt.show()

## 6. Interest Cost Pareto Chart

In [ ]:
summary_sorted = summary.sort_values('Interest_Cost', ascending=False)
cumulative_pct = summary_sorted['Interest_Cost'].cumsum() / summary_sorted['Interest_Cost'].sum() * 100

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(summary_sorted['Customer_Name'], summary_sorted['Interest_Cost'],
        color='#1d4ed8', edgecolor='white', width=0.5)
ax1.set_ylabel('Interest Cost (₹)')
ax1.set_title('Interest Cost Distribution (Pareto Chart)', fontsize=12, fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(summary_sorted['Customer_Name'], cumulative_pct,
         color='#f97316', marker='o', linewidth=2)
ax2.set_ylabel('Cumulative %')
ax2.set_ylim(0, 110)
ax2.axhline(y=80, color='red', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.savefig('../dashboards/interest_pareto.png', dpi=150)
plt.show()

## ✅ Credit Analysis Complete

**Key Findings:**
- Overall credit is **profitable** (total earnings ₹20,390 at 35% margin)
- **Kishore** is the only loss-making customer (−₹363) due to skipped repayments Feb 6–14
- **Anand** is the best credit customer (₹6,205 earnings)
- **Recommendation:** Set credit cap + enforce repayment schedule especially for Kishore

---
**Namra Sania | 23f2000313 | IIT Madras BDM Capstone**